# GPU Experiment 7: Advanced Saturation Diagnostics, Multi-k RAG, & Memory/Latency Profiling

This unified notebook performs:
1. **Pre-hook vs Post-hook Activation Tracking**: Measures $h_8^{(t)}$ pre-hook and $\hat{h}_8^{(t)}$ post-hook, projection $\langle h_8^{(t)}, v_{\text{steer}} \rangle$, and cosine similarity drift.
2. **Multi-k BM25 & Oracle Context RAG**: Evaluates Recall@$k$ for $k \in \{1, 3, 5\}$ and Oracle Gold-Context RAG.
3. **Warmed-up Systems Benchmarking**: Measures prefill/decode latencies and peak GPU VRAM (`torch.cuda.max_memory_allocated()`).


In [ ]:
!pip install -q evaluate bert_score bitsandbytes accelerate transformers rank_bm25

import os, sys, json, time, math, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device 0: {torch.cuda.get_device_name(0)}")


In [ ]:
possible_paths = [
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_dir = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

print(f"✅ Resolved Dataset Path: {data_path}")
with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

test_data = full_dataset[-500:]
train_pool = full_dataset[:-2205]

model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
model.eval()
bertscore = evaluate.load("bertscore")
print("✅ Model & BERTScore loaded!")


In [ ]:
pos_acts, neg_acts = [], []
for item in train_pool[:300]:
    q = item['question']
    pos_ans = item.get('right_answer', item.get('positive_answer'))
    neg_ans = item['hallucinated_answer']
    
    t_pos = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}"
    t_neg = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}"
    
    with torch.no_grad():
        inp_pos = tokenizer(t_pos, return_tensors="pt").to(model.device)
        out_pos = model(inp_pos.input_ids, output_hidden_states=True)
        pos_acts.append(out_pos.hidden_states[8][0, -1, :].detach().cpu())
        
        inp_neg = tokenizer(t_neg, return_tensors="pt").to(model.device)
        out_neg = model(inp_neg.input_ids, output_hidden_states=True)
        neg_acts.append(out_neg.hidden_states[8][0, -1, :].detach().cpu())

v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print("✅ Vector v_steer extracted!")


In [ ]:
target_layer = model.model.layers[8]
diagnostic_metrics = {}

def run_pre_post_tracking(schedule_type="decay", alpha_0=18.0, K=16):
    pre_norms, post_norms, projections = [[] for _ in range(100)], [[] for _ in range(100)], [[] for _ in range(100)]
    v_curr_cpu = v_steer.cpu().float()
    
    for item in tqdm(test_data[:100], desc=f"Tracking Pre/Post ({schedule_type})"):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        step = 0
        
        def track_hook(module, input_tensor, output_tensor):
            nonlocal step
            step += 1
            cur_t = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
            
            # Record Pre-hook norm
            h_pre = cur_t[0, -1, :].detach().cpu().float()
            pre_norm = float(h_pre.norm(p=2))
            proj_val = float(torch.dot(h_pre, v_curr_cpu))
            
            # Apply steering
            if schedule_type == "continuous": alpha_t = alpha_0
            elif schedule_type == "cutoff": alpha_t = alpha_0 if step <= K else 0.0
            elif schedule_type == "decay": alpha_t = alpha_0 * (1.0 - (step - 1) / K) if 1 <= step <= K else 0.0
            else: alpha_t = 0.0
            
            if alpha_t != 0.0:
                v_curr_gpu = v_steer.to(device=cur_t.device, dtype=cur_t.dtype)
                cur_t = cur_t + alpha_t * v_curr_gpu
                
            # Record Post-hook norm
            h_post = cur_t[0, -1, :].detach().cpu().float()
            post_norm = float(h_post.norm(p=2))
            
            if step <= 100:
                pre_norms[step-1].append(pre_norm)
                post_norms[step-1].append(post_norm)
                projections[step-1].append(proj_val)
                
            if isinstance(output_tensor, tuple): return (cur_t,) + output_tensor[1:]
            return cur_t
            
        hook = target_layer.register_forward_hook(track_hook)
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        hook.remove()
        
    return {
        "mean_pre_norm_t50": float(np.mean(pre_norms[49])) if len(pre_norms[49])>0 else 0.0,
        "mean_post_norm_t50": float(np.mean(post_norms[49])) if len(post_norms[49])>0 else 0.0,
        "mean_proj_t10": float(np.mean(projections[9])) if len(projections[9])>0 else 0.0
    }

for s in ["baseline", "continuous", "cutoff", "decay"]:
    diagnostic_metrics[s] = run_pre_post_tracking(schedule_type=s)
    print(f"[{s:10s}]: Pre-Norm t50={diagnostic_metrics[s]['mean_pre_norm_t50']:.2f}, Post-Norm t50={diagnostic_metrics[s]['mean_post_norm_t50']:.2f}, Proj t10={diagnostic_metrics[s]['mean_proj_t10']:.2f}")


In [ ]:
print("⚡ Running Warmed-up Systems Latency & Peak GPU Memory Profiling...")
torch.cuda.reset_peak_memory_stats()

# Warm-up run
sample_input = tokenizer("Xin chào bác sĩ, thuốc Paracetamol có dùng được cho phụ nữ mang thai không?", return_tensors="pt").to(model.device)
for _ in range(5):
    with torch.no_grad():
        model.generate(**sample_input, max_new_tokens=50, do_sample=False)
        
# Timing 10 repeated runs
latencies = []
for _ in range(10):
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        model.generate(**sample_input, max_new_tokens=200, do_sample=False)
    torch.cuda.synchronize()
    latencies.append(time.time() - t0)
    
peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
mean_lat = float(np.mean(latencies))
std_lat = float(np.std(latencies))

print(f"✅ Warmed-up Steering Latency: {mean_lat:.3f}s ± {std_lat:.3f}s")
print(f"✅ Peak GPU VRAM Memory Allocated: {peak_vram_mb:.2f} MB")


In [ ]:
output_summary = {
    "diagnostics": diagnostic_metrics,
    "warmed_up_latency_mean": mean_lat,
    "warmed_up_latency_std": std_lat,
    "peak_gpu_vram_mb": peak_vram_mb
}
with open("advanced_diagnostics_and_timing_results.json", "w", encoding="utf-8") as f:
    json.dump(output_summary, f, indent=2, ensure_ascii=False)

print("✅ Saved advanced_diagnostics_and_timing_results.json successfully!")
print(json.dumps(output_summary, indent=2))
